# Merge ETPL into the main dataset (CS + PSEO + IPEDS + BLS)

### Load data


In [13]:
import pandas as pd
import numpy as np
import re, os

MAIN_PATH = "../data/merges/CS_PSEO_IPEDS_BLS_merged.csv"
ETPL_PATH = "../data/processed/ETPL/ETPL_cleaned.csv"
OUTPUT_DIR = "../data/merges"

main = pd.read_csv(MAIN_PATH, low_memory=False).copy()
etpl = pd.read_csv(ETPL_PATH, low_memory=False)

print("Main:", main.shape)
print("ETPL:", etpl.shape)

Main: (51993, 200)
ETPL: (31621, 56)


### Fix duplicated rows in the main dataset

The main file should have **one row per program (UNITID + CIPCODE)**, but it has more rows than that.
The IPEDS program-level table lists each school's up to 6 largest programs, and it was joined on **UNITID only**,
so each Scorecard program got copied once per IPEDS program, even when the CIP codes don't match
(e.g. a school's CIP 1104 program paired with that school's welding program, 48.0508).

In [14]:
key = ["UNITID", "CIPCODE"]
print("Rows:", len(main), "| Unique programs (UNITID + CIPCODE):", main[key].drop_duplicates().shape[0])

# IPEDS program-level columns that came from the bad join (they differ between the copies)
ipeds_prog_cols = ["cip_code", "prog_tuition_fees", "prog_supplies", "prog_length", "length_unit_code",
                   "months_to_complete", "weeks_to_complete", "program_slot", "cip_2digit", "length_unit",
                   "total_prog_cost", "cost_per_hour", "hours_per_month", "prog_supplies_missing"]

# Does the IPEDS program's CIP actually match this row's CIP? (12.0401 -> 1204)
ipeds_cip4 = np.floor((main["cip_code"] * 100).round(4))
main["ipeds_prog_match"] = main["program_slot"].notna() & (ipeds_cip4 == main["CIPCODE"])

# For each program keep ONE row:
#   1) a row whose IPEDS program matches the CIP (lowest slot = largest program), otherwise
#   2) any row, with the IPEDS program columns blanked out (they belong to a different program)
main = main.sort_values(key + ["ipeds_prog_match", "program_slot"], ascending=[True, True, False, True])
main = main.drop_duplicates(subset=key, keep="first").copy()
main.loc[~main["ipeds_prog_match"], ipeds_prog_cols] = np.nan

print("After fix:", main.shape)
print("Programs with a matching IPEDS program record:", main["ipeds_prog_match"].sum())
assert main[key].duplicated().sum() == 0, "Still duplicated!"

Rows: 51993 | Unique programs (UNITID + CIPCODE): 26678
After fix: (26678, 201)
Programs with a matching IPEDS program record: 3644


### Build ETPL field-of-study benchmarks (national, by CIP4)

In [15]:
MIN_N = 5   # minimum programs needed to report a median

# Same CIP format as main: 4-digit integer (e.g. "51.09" -> 5109, "01.00" -> 100)
etpl["CIPCODE"] = etpl["cip4_nodot"].astype(int)

metrics = {
    "completion_rate": "completion",
    "employment_q2_rate": "employment_q2",
    "median_earnings_annual": "earnings",
    "total_contact_hours": "contact_hours",
    "tuition_capped": "tuition",
}

def summarize(group_cols, prefix):
    g = etpl.groupby(group_cols)
    out = pd.DataFrame({
        f"{prefix}_n_programs": g.size(),
        f"{prefix}_n_with_outcomes": g["has_outcomes"].sum(),
        f"{prefix}_share_wp_length": g["wp_length_ok"].mean(),       # share of programs in the Workforce Pell length window
    })
    for col, name in metrics.items():
        med = g[col].median()
        n = g[col].count()
        out[f"{prefix}_{name}_median"] = med.where(n >= MIN_N)     # blank if fewer than MIN_N programs
    return out.reset_index()

etpl_natl = summarize(["CIPCODE"], "etpl_natl")
print("CIP4 fields with ETPL benchmarks:", len(etpl_natl))
etpl_natl.head()

CIP4 fields with ETPL benchmarks: 301


,CIPCODE,etpl_natl_n_programs,etpl_natl_n_with_outcomes,etpl_natl_share_wp_length,etpl_natl_completion_median,etpl_natl_employment_q2_median,etpl_natl_earnings_median,etpl_natl_contact_hours_median,etpl_natl_tuition_median
0,100,8,0,0.125000,NaN,NaN,NaN,194.0,3489.5
1,101,79,27,0.025316,0.846154,0.670068,29995.20,480.0,3515.0
2,102,27,7,0.074074,0.727273,1.000000,35012.00,1054.0,6000.0
3,103,56,10,0.053571,0.699324,0.652926,27369.12,448.0,3339.0
4,104,2,2,0.000000,NaN,NaN,NaN,NaN,NaN


### Build ETPL state benchmarks (CIP4 + state), with national fallback
Local labor markets differ, so a state benchmark is more relevant when there's enough data.
If a state has fewer than 5 programs for that field, we fall back to the national number and record which level was used.

In [16]:
etpl_state = summarize(["CIPCODE", "state"], "etpl_state")
print("CIP4 x state groups:", len(etpl_state))
etpl_state.head()

CIP4 x state groups: 3501


,CIPCODE,state,etpl_state_n_programs,etpl_state_n_with_outcomes,etpl_state_share_wp_length,etpl_state_completion_median,etpl_state_employment_q2_median,etpl_state_earnings_median,etpl_state_contact_hours_median,etpl_state_tuition_median
0,100,IN,1,0,0.000000,NaN,NaN,NaN,NaN,NaN
1,100,MO,3,0,0.333333,NaN,NaN,NaN,NaN,NaN
2,100,MT,1,0,0.000000,NaN,NaN,NaN,NaN,NaN
3,100,NE,1,0,0.000000,NaN,NaN,NaN,NaN,NaN
4,100,OH,2,0,0.000000,NaN,NaN,NaN,NaN,NaN


### Direct program match (same school + same CIP4)

ETPL has no UNITID, so we match **cleaned school name + state** to IPEDS names in the main file
(same cleaning rules used in the ETPL cleaning notebook). This is an exact match on the cleaned name,
so some real matches are missed; it's a conservative starting point.

In [17]:
ABBREVIATIONS = {r"\bcomm\b": "community", r"\bcc\b": "community college", r"\bcoll\b": "college",
                 r"\buniv\b": "university", r"\btech\b": "technical", r"\binst\b": "institute",
                 r"\bst\b": "state", r"&": " and "}

def clean_name(name):
    if pd.isna(name):
        return np.nan
    n = re.sub(r"[^\w\s&]", " ", str(name).lower())
    for pattern, repl in ABBREVIATIONS.items():
        n = re.sub(pattern, repl, n)
    n = re.sub(r"\b(the|inc|llc|corp)\b", " ", n)
    return re.sub(r"\s+", " ", n).strip()

# School lookup from the main file: cleaned name + state -> UNITID
schools = main[["UNITID", "INSTNM", "state"]].drop_duplicates("UNITID").copy()
schools["provider_match_key"] = schools["INSTNM"].map(clean_name) + "|" + schools["state"]
schools = schools.drop_duplicates("provider_match_key", keep=False)   # drop ambiguous names (2 schools, same key)

etpl_m = etpl.merge(schools[["provider_match_key", "UNITID"]], on="provider_match_key", how="inner")
print("ETPL providers matched to a school:", etpl_m["provider_unique_id"].nunique(),
      "of", etpl["provider_unique_id"].nunique())

# One school can list several ETPL programs in the same CIP4 -> combine them
g = etpl_m.groupby(["UNITID", "CIPCODE"])
etpl_prog = pd.DataFrame({
    "etpl_prog_n_programs": g.size(),
    "etpl_prog_completion": g["completion_rate"].median(),
    "etpl_prog_employment_q2": g["employment_q2_rate"].median(),
    "etpl_prog_earnings": g["median_earnings_annual"].median(),
    "etpl_prog_contact_hours": g["total_contact_hours"].median(),
    "etpl_prog_tuition": g["tuition_capped"].median(),
    "etpl_prog_wp_length_ok": g["wp_length_ok"].any(),
    "etpl_prog_likely_wp": g["likely_workforce_pell"].any(),
}).reset_index()
print("UNITID + CIP4 pairs with an ETPL program:", len(etpl_prog))

ETPL providers matched to a school: 710 of 5103
UNITID + CIP4 pairs with an ETPL program: 4698


## Merge everything into the main dataset

In [18]:
n_before = len(main)
merged = (main
          .merge(etpl_natl,  on="CIPCODE", how="left")
          .merge(etpl_state, on=["CIPCODE", "state"], how="left")
          .merge(etpl_prog,  on=["UNITID", "CIPCODE"], how="left"))

# Safety check: a left join on unique keys must NOT add rows
assert len(merged) == n_before, f"Row count changed: {n_before} -> {len(merged)}"

# Best available benchmark: state if enough data, otherwise national
for name in ["completion", "employment_q2", "earnings", "contact_hours", "tuition"]:
    st, nat = merged[f"etpl_state_{name}_median"], merged[f"etpl_natl_{name}_median"]
    merged[f"etpl_bench_{name}"] = st.fillna(nat)
merged["etpl_bench_level"] = np.select(
    [merged["etpl_state_earnings_median"].notna(), merged["etpl_natl_earnings_median"].notna()],
    ["state", "national"], default="none")

merged["has_etpl_field_match"] = merged["etpl_natl_n_programs"].notna()
merged["has_etpl_program_match"] = merged["etpl_prog_n_programs"].notna()
merged["has_etpl_program_outcomes"] = merged[["etpl_prog_completion", "etpl_prog_employment_q2",
                                              "etpl_prog_earnings"]].notna().any(axis=1)

print("Final shape:", merged.shape)
print(f"Programs with an ETPL field benchmark:   {merged['has_etpl_field_match'].mean():.1%}")
print(f"Programs with a direct ETPL match:       {merged['has_etpl_program_match'].mean():.1%}")
print(f"  ...with direct ETPL outcomes:          {merged['has_etpl_program_outcomes'].mean():.1%}")
print("\nBenchmark level used:")
print(merged["etpl_bench_level"].value_counts().to_string())

Final shape: (26678, 234)
Programs with an ETPL field benchmark:   97.9%
Programs with a direct ETPL match:       13.0%
  ...with direct ETPL outcomes:          8.3%

Benchmark level used:
etpl_bench_level
national    12668
state        9595
none         4415


### Local (state-level) earnings adjustment

In [19]:
# Workforce Pell adjusts earnings for local conditions, not national numbers.
# Without BEA price data, we use ETPL to measure how much each state earns
# compared to the national median FOR THE SAME FIELD (CIP4).

MIN_STATE_N = 30        # states need at least 30 ETPL programs with earnings, otherwise use region
FPL_150 = 23_475        # 150% of the federal poverty line, single person, 2025

# 1) Each ETPL program's earnings vs. the national median for its field
e = etpl.dropna(subset=["median_earnings_annual"]).copy()
field_natl = e.groupby("CIPCODE")["median_earnings_annual"].median().rename("field_natl_median")
e = e.join(field_natl, on="CIPCODE")
e["ratio_to_natl"] = e["median_earnings_annual"] / e["field_natl_median"]

# 2) State index = typical ratio in that state (1.10 = earns 10% more than national for the same field)
state_idx = e.groupby("state")["ratio_to_natl"].agg(["median", "count"])
state_idx = state_idx.loc[state_idx["count"] >= MIN_STATE_N, "median"].rename("local_earnings_index_state")

# 3) Region index as a fallback for states with too few programs
state_region = merged[["state", "bea_region"]].drop_duplicates("state")
e = e.merge(state_region, on="state", how="left")
region_idx = e.groupby("bea_region")["ratio_to_natl"].median().rename("local_earnings_index_region")

merged = merged.join(state_idx, on="state").join(region_idx, on="bea_region")
merged["local_earnings_index"] = (merged["local_earnings_index_state"]
                                  .fillna(merged["local_earnings_index_region"])
                                  .fillna(1.0))
merged["local_index_level"] = np.select(
    [merged["local_earnings_index_state"].notna(), merged["local_earnings_index_region"].notna()],
    ["state", "region"], default="national")

# 4) Locally adjusted earnings and the Workforce Pell value-added test
merged["earnings_1yr"] = merged["EARN_MDN_1YR"].fillna(merged["y1_earnings_program"])   # Scorecard, else PSEO
merged["earnings_1yr_local_adj"] = merged["earnings_1yr"] / merged["local_earnings_index"]
merged["value_added_earnings_local"] = merged["earnings_1yr_local_adj"] - FPL_150
merged["program_cost"] = merged["total_prog_cost"].fillna(merged["py_tuitfee"])

# Pass/fail flags (blank when the data needed isn't available)
merged["vae_positive_local"] = np.where(merged["value_added_earnings_local"].notna(),
                                        merged["value_added_earnings_local"] > 0, np.nan)
has_both = merged["value_added_earnings_local"].notna() & merged["program_cost"].notna()
merged["passes_vae_local"] = np.where(has_both,
                                      merged["value_added_earnings_local"] >= merged["program_cost"], np.nan)

# 5) How does the program compare to the same field in its state? (>1 = beats the local benchmark)
merged["earnings_vs_local_field"] = merged["earnings_1yr"] / merged["etpl_bench_earnings"]

print(merged["local_index_level"].value_counts().to_string())
nat = (merged["earnings_1yr"] > FPL_150)[merged["earnings_1yr"].notna()].mean()
loc = merged["vae_positive_local"].dropna().mean()
print(f"\nValue-added > 0  national: {nat:.1%}  | local-adjusted: {loc:.1%}")
flip = (merged["earnings_1yr"] > FPL_150) != (merged["value_added_earnings_local"] > 0)
print("Programs whose result changes with local adjustment:", int(flip[merged["earnings_1yr"].notna()].sum()))

local_index_level
state     24411
region     2267

Value-added > 0  national: 77.8%  | local-adjusted: 78.3%
Programs whose result changes with local adjustment: 448


**Notes:**

- Workforce Pell adjusts earnings for local conditions, so we added a **local earnings index** built from ETPL: each state's earnings compared to the national median **for the same field**.
- States with 30+ ETPL programs use their own index (24,411 programs). Smaller states use their region (2,267 programs).
- Index ranges from **0.46 (PR)** to **1.20 (CA)**.
- New fields: locally adjusted earnings, local value-added earnings (adjusted earnings − $23,475), and pass/fail flags for the value-added test.
- **448 programs change pass/fail** compared with using national numbers.
- Caveat: this is an earnings index, used as a stand-in for BEA regional price parities (which the official rule uses).

## do ETPL and Scorecard agree?

In [20]:
both = merged.dropna(subset=["etpl_prog_earnings", "EARN_MDN_1YR"])
print("Programs with BOTH direct ETPL earnings and Scorecard 1-yr earnings:", len(both))
if len(both) > 10:
    print("Correlation:", round(both["etpl_prog_earnings"].corr(both["EARN_MDN_1YR"]), 2))
    print("Median ETPL earnings:     ", f"${both['etpl_prog_earnings'].median():,.0f}")
    print("Median Scorecard earnings:", f"${both['EARN_MDN_1YR'].median():,.0f}")

f = merged.dropna(subset=["etpl_bench_earnings", "EARN_MDN_1YR"])
print("\nField benchmark vs Scorecard earnings correlation:", round(f["etpl_bench_earnings"].corr(f["EARN_MDN_1YR"]), 2),
      f"(n={len(f):,})")

Programs with BOTH direct ETPL earnings and Scorecard 1-yr earnings: 536
Correlation: 0.44
Median ETPL earnings:      $32,691
Median Scorecard earnings: $33,892

Field benchmark vs Scorecard earnings correlation: 0.46 (n=4,631)


**Notes:**

- **536 programs** have both direct ETPL earnings and Scorecard 1-year earnings, so we can compare the two sources for the same programs.
- **The typical earnings are almost the same:** ETPL median **$32,691** vs. Scorecard median **$33,892** (about 4% apart). This shows the name + CIP matching is linking the right programs.
- **The correlation is moderate (0.44), and that's expected.** The two sources measure earnings differently:
  - ETPL = earnings in the **2nd quarter after leaving**, × 4. Scorecard = earnings **1 year after completing**.
  - ETPL counts **everyone who left** the program, including people who didn't finish. Scorecard only counts completers.
  - Different student groups and years
  - A CIP4 match can combine several ETPL programs into one
- **Field benchmark vs. Scorecard: 0.46 correlation (n = 4,631).** Field-level ETPL earnings track program earnings about as well as the direct matches do. That supports using `etpl_bench_*` as features for the 98% of programs without a direct match.
- **Takeaway:** ETPL agrees with Scorecard on typical earnings but isn't a copy of it, so it adds new information to the model instead of repeating what's already there.

--------

In [21]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = f"{OUTPUT_DIR}/CS_PSEO_IPEDS_BLS_ETPL_merged.csv"
merged.to_csv(out_path, index=False)
print("Saved:", out_path, merged.shape)

etpl_cols = [c for c in merged.columns if c.startswith("etpl_") or "etpl" in c]
print(f"\n{len(etpl_cols)} ETPL columns added:")
print(etpl_cols)

Saved: ../data/merges/CS_PSEO_IPEDS_BLS_ETPL_merged.csv (26678, 245)

33 ETPL columns added:
['etpl_natl_n_programs', 'etpl_natl_n_with_outcomes', 'etpl_natl_share_wp_length', 'etpl_natl_completion_median', 'etpl_natl_employment_q2_median', 'etpl_natl_earnings_median', 'etpl_natl_contact_hours_median', 'etpl_natl_tuition_median', 'etpl_state_n_programs', 'etpl_state_n_with_outcomes', 'etpl_state_share_wp_length', 'etpl_state_completion_median', 'etpl_state_employment_q2_median', 'etpl_state_earnings_median', 'etpl_state_contact_hours_median', 'etpl_state_tuition_median', 'etpl_prog_n_programs', 'etpl_prog_completion', 'etpl_prog_employment_q2', 'etpl_prog_earnings', 'etpl_prog_contact_hours', 'etpl_prog_tuition', 'etpl_prog_wp_length_ok', 'etpl_prog_likely_wp', 'etpl_bench_completion', 'etpl_bench_employment_q2', 'etpl_bench_earnings', 'etpl_bench_contact_hours', 'etpl_bench_tuition', 'etpl_bench_level', 'has_etpl_field_match', 'has_etpl_program_match', 'has_etpl_program_outcomes']
